In [5]:
import numpy as np

from weather.config import (
    Experiment,
    WeatherFixedParams,
    WeatherGridParams,
    MLPFixedParams,
    MLPGridParams,
    FitFixedParams,
    FitGridParams,
)

from weather.search import Search

from mlp.utils import (
    plot_loss,
    regression_report,
    classification_report_binary,
    plot_roc_auc,
    plot_accuracy,
    accuracy_within_tolerance,
)


In [6]:
SEED = 42
np.random.seed(SEED)


In [7]:
# =========================================================
# 1) TEMPERATURE REGRESSION
# =========================================================
exp_temp_encoding = Experiment(
    name="temperature_regression_encoding",

    # =========================
    # WEATHER
    # =========================
    weather_fixed=WeatherFixedParams(
        target="temperature",
        target_mode="regression",
        # target_threshold=6.0,
        data_dir="../data",
        skip_day=True,
        # normalization="global",
        encode_wind_direction=True,
    ),

    weather_grid=WeatherGridParams(
        window_aggregation="flatten",
        window_size=[1,2,3,4,5,6,7,8,9,10],
        normalization="standardize",
        input_variables=("temperature",),
        aggregations={
            "temperature": ("mean", "min", "max"),
            "humidity": ("mean", "min", "max"),
            "pressure": ("mean", "min", "max"),
            "wind_speed": ("mean", "max"),
            "wind_direction": ("mean",),
        },
        cities=("Vancouver",),
    ),

    # =========================
    # MLP
    # =========================
    mlp_fixed=MLPFixedParams(
        task="regression",
        beta = 0.9,
        beta2 = 0.999,
        eps = 1e-8,
        adaptive_lr=True,
        lr_decay=0.99,
    ),

    mlp_grid=MLPGridParams(
        hidden_layers=(32, 64),
        loss="huber",
        activation="gelu",
        learning_rate=0.01,
        seed=SEED,
        use_bias=True,
        optimizer="momentum",
    ),

    # =========================
    # FIT
    # =========================
    fit_fixed=FitFixedParams(
        verbose=True,
        log_every=None,
        use_tqdm=True,
        one_hot_if_needed=True,
        early_stopping=True,
        patience=50,
        min_delta=0.0001,
    ),

    fit_grid=FitGridParams(
        epochs=400,
        batch_size="auto",
        shuffle=False,
        val_split=0.1,
    ),
)

# =========================================================
# 2) WIND (>=6 m/s) BINARY CLASSIFICATION
# =========================================================
exp_wind_encoding = Experiment(
    name="wind6_binary_encoding",

    # =========================
    # WEATHER
    # =========================
    weather_fixed=WeatherFixedParams(
        target="wind_speed",
        target_mode="binary",
        target_threshold=6.0,
        data_dir="../data",
        skip_day=True,
        # normalization="global",
        encode_wind_direction=True,
    ),

    weather_grid=WeatherGridParams(
        window_aggregation="flatten",
        window_size=[1,2,3,4,5,6,7,8,9,10],
        normalization="standardize",
        input_variables=("wind_speed",),
        aggregations={
            "temperature": ("mean", "min", "max"),
            "humidity": ("mean", "min", "max"),
            "pressure": ("mean", "min", "max"),
            "wind_speed": ("mean", "max"),
            "wind_direction": ("mean",),
        },
        cities=("Vancouver",),
    ),

    # =========================
    # MLP
    # =========================
    mlp_fixed=MLPFixedParams(
        task="binary",
        beta = 0.9,
        beta2 = 0.999,
        eps = 1e-8,
        adaptive_lr=True,
        lr_decay=0.99,
    ),

    mlp_grid=MLPGridParams(
        hidden_layers=(64, 32),
        loss="binary_cross_entropy",
        activation="gelu",
        learning_rate=0.01,
        seed=SEED,
        use_bias=True,
        optimizer="momentum",
    ),

    # =========================
    # FIT
    # =========================
    fit_fixed=FitFixedParams(
        verbose=True,
        log_every=None,
        use_tqdm=True,
        one_hot_if_needed=True,
        early_stopping=True,
        patience=50,
        min_delta=0.0001,
    ),

    fit_grid=FitGridParams(
        epochs=400,
        batch_size="auto",
        shuffle=False,
        val_split=0.1,
    ),
)

experiments = [exp_temp_encoding, exp_wind_encoding]


In [8]:
search = Search()

results1 = search.run(exp_temp_encoding)



Starting experiment: temperature_regression_encoding

Building dataset
  → TRAIN split


Vancouver | windows: 100%|██████████| 1520/1520 [00:01<00:00, 1401.87it/s]


  → TEST split


Vancouver | windows: 100%|██████████| 363/363 [00:00<00:00, 1524.53it/s]



Configuration run 1/10:
WEATHER (variable):
  - window_size: 1

Training model


...

  → TEST split


Vancouver | windows: 100%|██████████| 354/354 [00:01<00:00, 199.37it/s]



Configuration run 10/10:
WEATHER (variable):
  - window_size: 10

Training model


Training:  51%|█████▏    | 205/400 [00:04<00:04, 42.20it/s, acc=n/a, loss=1.4693, lr=0.00127413]

Early stopping at epoch 206, best val_loss=1.346276 after 50 epochs without improvement.
Training finished in 4.86 seconds

Experiment finished | total runs = 10



In [13]:
from IPython.core.display import HTML

for run in results1:
    model = run["model"]
    y_test = run["y_test"]
    y_pred = run["y_pred"]
    y_proba = run["y_proba"]

    history = run["history"]
    accuracy_history = run["accuracy_history"]
    config_log = run.get("config_log", {})

    print("\n\n" + "=" * 80)
    print(f"EXPERIMENT: {run['experiment']}")
    print(f"TASK: {model.task.upper()}")
    print("=" * 80)

    # ========= CONFIGURATION (VARIABLE PARAMS) =========
    if config_log:
        print("CONFIGURATION (variable params):")
        for section, params in config_log.items():
            if not params:
                continue
            print(f"  {section.upper()}:")
            for k, v in params.items():
                print(f"    - {k}: {v}")
        print("-" * 80)

    # ===================== METRICS =====================
    if model.task == "binary":
        metrics = classification_report_binary(
            y_true=y_test,
            y_score=y_proba,
            threshold=0.5,
        )

        auc_val = metrics["auc"]
        if auc_val >= 0.65:
            auc_color = "#2e7d32"   # dark green
        elif auc_val >= 0.60:
            auc_color = "#558b2f"   # olive green
        elif auc_val >= 0.58:
            auc_color = "#f9a825"   # amber
        elif auc_val >= 0.55:
            auc_color = "#ef6c00"   # orange
        else:
            auc_color = "#c62828"   # red

        print("=== TEST METRICS (BINARY CLASSIFICATION) ===")
        print(f"Accuracy : {metrics['accuracy']:.4f}")
        print(f"Precision: {metrics['precision']:.4f}")
        print(f"Recall   : {metrics['recall']:.4f}")
        print(f"AUC      : {metrics['auc']:.4f}")
        display(HTML(
            f"""
            <div style="
                font-family: 'JetBrains Mono', 'Consolas', 'Menlo', monospace;
                font-size: 13px;
                color: {auc_color};
                padding-left: 12px;
                margin: 4px 0;
            ">
                <b>AUC</b>: {auc_val:.4f}
            </div>
            """
        ))

    else:
        metrics = regression_report(
            y_true=y_test,
            y_pred=y_pred,
        )

        acc_2 = accuracy_within_tolerance(y_test, y_pred, tol=2.0)
        acc_25 = accuracy_within_tolerance(y_test, y_pred, tol=2.5)

        if acc_2 >= 0.62:
            color = "#2e7d32"   # dark green
        elif acc_2 >= 0.61:
            color = "#558b2f"   # olive green
        elif acc_2 >= 0.60:
            color = "#f9a825"   # amber
        elif acc_2 >= 0.58:
            color = "#ef6c00"   # orange
        else:
            color = "#c62828"   # red

        print("=== TEST METRICS (REGRESSION) ===")
        print(f"MAE              : {metrics['mae']:.4f}")
        print(f"MSE              : {metrics['mse']:.4f}")
        print(f"RMSE             : {metrics['rmse']:.4f}")
        display(HTML(
            f"""
            <div style="
                font-family: 'JetBrains Mono', 'Consolas', 'Menlo', monospace;
                font-size: 13px;
                color: {color};
                padding-left: 12px;
                margin: 4px 0;
            ">
                <b>Accuracy |err|≤2°C</b>: {acc_2:.4f}
            </div>
            """
        ))
        print(f"Accuracy |err|≤2°C   : {acc_2:.4f}")

    # --- plots ---
    # plot_loss(
    #     history,
    #     title="Train Loss Evolution",
    # )
    #
    # if model.task != "regression":
    #     if accuracy_history and not all(np.isnan(accuracy_history)):
    #         plot_accuracy(
    #             accuracy_history,
    #             title="Accuracy evolution",
    #         )
    #
    # # ROC only for binary classification
    # if model.task == "binary":
    #     plot_roc_auc(
    #         y_true=y_test,
    #         y_score=y_proba,
    #         title="ROC Curve",
    #     )




EXPERIMENT: temperature_regression_encoding
TASK: REGRESSION
CONFIGURATION (variable params):
  WEATHER:
    - window_size: 1
--------------------------------------------------------------------------------
=== TEST METRICS (REGRESSION) ===
MAE              : 2.3761
MSE              : 12.8090
RMSE             : 3.5790


Accuracy |err|≤2°C   : 0.5606


EXPERIMENT: temperature_regression_encoding
TASK: REGRESSION
CONFIGURATION (variable params):
  WEATHER:
    - window_size: 2
--------------------------------------------------------------------------------
=== TEST METRICS (REGRESSION) ===
MAE              : 1.9629
MSE              : 6.7382
RMSE             : 2.5958


Accuracy |err|≤2°C   : 0.6049


EXPERIMENT: temperature_regression_encoding
TASK: REGRESSION
CONFIGURATION (variable params):
  WEATHER:
    - window_size: 3
--------------------------------------------------------------------------------
=== TEST METRICS (REGRESSION) ===
MAE              : 1.9234
MSE              : 6.4342
RMSE             : 2.5366


Accuracy |err|≤2°C   : 0.6189


EXPERIMENT: temperature_regression_encoding
TASK: REGRESSION
CONFIGURATION (variable params):
  WEATHER:
    - window_size: 4
--------------------------------------------------------------------------------
=== TEST METRICS (REGRESSION) ===
MAE              : 2.0314
MSE              : 7.2657
RMSE             : 2.6955


Accuracy |err|≤2°C   : 0.6086


EXPERIMENT: temperature_regression_encoding
TASK: REGRESSION
CONFIGURATION (variable params):
  WEATHER:
    - window_size: 5
--------------------------------------------------------------------------------
=== TEST METRICS (REGRESSION) ===
MAE              : 1.9103
MSE              : 6.1099
RMSE             : 2.4718


...

Accuracy |err|≤2°C   : 0.6400


EXPERIMENT: temperature_regression_encoding
TASK: REGRESSION
CONFIGURATION (variable params):
  WEATHER:
    - window_size: 7
--------------------------------------------------------------------------------
=== TEST METRICS (REGRESSION) ===
MAE              : 2.1505
MSE              : 8.5225
RMSE             : 2.9193


Accuracy |err|≤2°C   : 0.5895


EXPERIMENT: temperature_regression_encoding
TASK: REGRESSION
CONFIGURATION (variable params):
  WEATHER:
    - window_size: 8
--------------------------------------------------------------------------------
=== TEST METRICS (REGRESSION) ===
MAE              : 2.0638
MSE              : 7.4078
RMSE             : 2.7217


Accuracy |err|≤2°C   : 0.5820


EXPERIMENT: temperature_regression_encoding
TASK: REGRESSION
CONFIGURATION (variable params):
  WEATHER:
    - window_size: 9
--------------------------------------------------------------------------------
=== TEST METRICS (REGRESSION) ===
MAE              : 2.5053
MSE              : 15.1869
RMSE             : 3.8970


Accuracy |err|≤2°C   : 0.5776


EXPERIMENT: temperature_regression_encoding
TASK: REGRESSION
CONFIGURATION (variable params):
  WEATHER:
    - window_size: 10
--------------------------------------------------------------------------------
=== TEST METRICS (REGRESSION) ===
MAE              : 2.8153
MSE              : 22.7018
RMSE             : 4.7646


Accuracy |err|≤2°C   : 0.5576


In [10]:
search = Search()

results2 = search.run(exp_wind_encoding)


Starting experiment: wind6_binary_encoding

Building dataset
  → TRAIN split


Vancouver | windows: 100%|██████████| 1520/1520 [00:01<00:00, 1426.51it/s]


  → TEST split


Vancouver | windows: 100%|██████████| 363/363 [00:00<00:00, 1137.68it/s]



Configuration run 1/10:
WEATHER (variable):
  - window_size: 1

Training model


...

  → TEST split


Vancouver | windows: 100%|██████████| 354/354 [00:01<00:00, 247.05it/s]



Configuration run 10/10:
WEATHER (variable):
  - window_size: 10

Training model


Training:  44%|████▍     | 178/400 [00:10<00:13, 16.45it/s, acc=0.6858, loss=0.5825, lr=0.00167134]

Early stopping at epoch 179, best val_loss=0.662789, train_acc=0.6858, val_acc=0.6316 after 50 epochs without improvement.
Training finished in 10.82 seconds

Experiment finished | total runs = 10



In [12]:
from IPython.core.display import HTML

for run in results2:
    model = run["model"]
    y_test = run["y_test"]
    y_pred = run["y_pred"]
    y_proba = run["y_proba"]

    history = run["history"]
    accuracy_history = run["accuracy_history"]
    config_log = run.get("config_log", {})

    print("\n\n" + "=" * 80)
    print(f"EXPERIMENT: {run['experiment']}")
    print(f"TASK: {model.task.upper()}")
    print("=" * 80)

    # ========= CONFIGURATION (VARIABLE PARAMS) =========
    if config_log:
        print("CONFIGURATION (variable params):")
        for section, params in config_log.items():
            if not params:
                continue
            print(f"  {section.upper()}:")
            for k, v in params.items():
                print(f"    - {k}: {v}")
        print("-" * 80)

    # ===================== METRICS =====================
    if model.task == "binary":
        metrics = classification_report_binary(
            y_true=y_test,
            y_score=y_proba,
            threshold=0.5,
        )

        auc_val = metrics["auc"]
        if auc_val >= 0.60:
            auc_color = "#2e7d32"   # dark green
        elif auc_val >= 0.55:
            auc_color = "#558b2f"   # olive green
        elif auc_val >= 0.53:
            auc_color = "#f9a825"   # amber
        elif auc_val >= 0.51:
            auc_color = "#ef6c00"   # orange
        else:
            auc_color = "#c62828"   # red

        print("=== TEST METRICS (BINARY CLASSIFICATION) ===")
        print(f"Accuracy : {metrics['accuracy']:.4f}")
        print(f"Precision: {metrics['precision']:.4f}")
        print(f"Recall   : {metrics['recall']:.4f}")
        print(f"Auc      : {metrics['auc']:.4f}")
        display(HTML(
            f"""
            <div style="
                font-family: 'JetBrains Mono', 'Consolas', 'Menlo', monospace;
                font-size: 13px;
                color: {auc_color};
                padding-left: 12px;
                margin: 4px 0;
            ">
                <b>AUC</b>: {auc_val:.4f}
            </div>
            """
        ))

    else:
        metrics = regression_report(
            y_true=y_test,
            y_pred=y_pred,
        )

        acc_2 = accuracy_within_tolerance(y_test, y_pred, tol=2.0)
        acc_25 = accuracy_within_tolerance(y_test, y_pred, tol=2.5)

        if acc_2 >= 0.62:
            color = "#2e7d32"   # dark green
        elif acc_2 >= 0.61:
            color = "#558b2f"   # olive green
        elif acc_2 >= 0.60:
            color = "#f9a825"   # amber
        elif acc_2 >= 0.58:
            color = "#ef6c00"   # orange
        else:
            color = "#c62828"   # red

        print("=== TEST METRICS (REGRESSION) ===")
        print(f"MAE              : {metrics['mae']:.4f}")
        print(f"MSE              : {metrics['mse']:.4f}")
        print(f"RMSE             : {metrics['rmse']:.4f}")
        display(HTML(
            f"""
            <div style="
                font-family: 'JetBrains Mono', 'Consolas', 'Menlo', monospace;
                font-size: 13px;
                color: {color};
                padding-left: 12px;
                margin: 4px 0;
            ">
                <b>Accuracy |err|≤2°C</b>: {acc_2:.4f}
            </div>
            """
        ))
        print(f"Accuracy |err|≤2°C : {acc_2:.4f}")

    # --- plots ---
    # plot_loss(
    #     history,
    #     title="Train Loss Evolution",
    # )
    #
    # if model.task != "regression":
    #     if accuracy_history and not all(np.isnan(accuracy_history)):
    #         plot_accuracy(
    #             accuracy_history,
    #             title="Accuracy evolution",
    #         )
    #
    # # ROC only for binary classification
    # if model.task == "binary":
    #     plot_roc_auc(
    #         y_true=y_test,
    #         y_score=y_proba,
    #         title="ROC Curve",
    #     )




EXPERIMENT: wind6_binary_encoding
TASK: BINARY
CONFIGURATION (variable params):
  WEATHER:
    - window_size: 1
--------------------------------------------------------------------------------
=== TEST METRICS (BINARY CLASSIFICATION) ===
Accuracy : 0.5909
Precision: 0.6224
Recall   : 0.7732
Auc      : 0.5466




EXPERIMENT: wind6_binary_encoding
TASK: BINARY
CONFIGURATION (variable params):
  WEATHER:
    - window_size: 2
--------------------------------------------------------------------------------
=== TEST METRICS (BINARY CLASSIFICATION) ===
Accuracy : 0.5015
Precision: 0.6058
Recall   : 0.4301
Auc      : 0.5385




EXPERIMENT: wind6_binary_encoding
TASK: BINARY
CONFIGURATION (variable params):
  WEATHER:
    - window_size: 3
--------------------------------------------------------------------------------
=== TEST METRICS (BINARY CLASSIFICATION) ===
Accuracy : 0.5732
Precision: 0.6204
Recall   : 0.6979
Auc      : 0.5682




EXPERIMENT: wind6_binary_encoding
TASK: BINARY
CONFIGURATION (variable params):
  WEATHER:
    - window_size: 4
--------------------------------------------------------------------------------
=== TEST METRICS (BINARY CLASSIFICATION) ===
Accuracy : 0.5443
Precision: 0.6071
Recall   : 0.6230
Auc      : 0.5387




EXPERIMENT: wind6_binary_encoding
TASK: BINARY
CONFIGURATION (variable params):
  WEATHER:
    - window_size: 5
--------------------------------------------------------------------------------
=== TEST METRICS (BINARY CLASSIFICATION) ===
Accuracy : 0.5153
Precision: 0.6039
Recall   : 0.4895
Auc      : 0.5387




EXPERIMENT: wind6_binary_encoding
TASK: BINARY
CONFIGURATION (variable params):
  WEATHER:
    - window_size: 6
--------------------------------------------------------------------------------
=== TEST METRICS (BINARY CLASSIFICATION) ===
Accuracy : 0.5138
Precision: 0.5800
Recall   : 0.6105
Auc      : 0.5212




EXPERIMENT: wind6_binary_encoding
TASK: BINARY
CONFIGURATION (variable params):
  WEATHER:
    - window_size: 7
--------------------------------------------------------------------------------
=== TEST METRICS (BINARY CLASSIFICATION) ===
Accuracy : 0.5000
Precision: 0.5659
Recall   : 0.6138
Auc      : 0.5233




EXPERIMENT: wind6_binary_encoding
TASK: BINARY
CONFIGURATION (variable params):
  WEATHER:
    - window_size: 8
--------------------------------------------------------------------------------
=== TEST METRICS (BINARY CLASSIFICATION) ===
Accuracy : 0.5232
Precision: 0.6076
Recall   : 0.5106
Auc      : 0.5432




EXPERIMENT: wind6_binary_encoding
TASK: BINARY
CONFIGURATION (variable params):
  WEATHER:
    - window_size: 9
--------------------------------------------------------------------------------
=== TEST METRICS (BINARY CLASSIFICATION) ===
Accuracy : 0.5280
Precision: 0.6159
Recall   : 0.4973
Auc      : 0.5493




EXPERIMENT: wind6_binary_encoding
TASK: BINARY
CONFIGURATION (variable params):
  WEATHER:
    - window_size: 10
--------------------------------------------------------------------------------
=== TEST METRICS (BINARY CLASSIFICATION) ===
Accuracy : 0.5265
Precision: 0.6000
Recall   : 0.5484
Auc      : 0.5355
